# Karthik-1 Strategy Notebook

**Team 6 — FE747 Quantitative Investment Strategies, Spring 2026**

A self-contained, reproducible implementation of the Karthik-1 dual-logit regime strategy.

---

## What this notebook does

1. **Loads** daily price data for six Vanguard funds (1990–2026).
2. **Builds features** (FF4-style factors + VIX proxy with 5-day and 21-day smoothing).
3. **Fits two L1 logits** — one for drawdown probability, one for upside probability — on the 1995-1998 fit window.
4. **Generates daily signals** at the chosen probability cutoffs (DD ≥ 15% @ 0.40, UP ≥ 8% @ 0.30).
5. **Runs the four-state allocation simulation** (90/55/15/85 equity weights, monthly rebalance) matching the FE747 sim engine spec.
6. **Reports diagnostics**: confusion matrices, equity curve, year-by-year alpha, state attribution.
7. **Exports** model JSONs and probability CSVs in the format the FE747 sim engine expects.

## Strategy at a glance

| Component | Choice |
|---|---|
| Equity asset | VFINX (Vanguard 500 Index) |
| Fixed income | VBMFX (Total Bond Market) |
| Fit window | 1995-01-01 → 1998-12-31 |
| DD threshold / cutoff | 15% / 0.40 |
| UP threshold / cutoff | 8% / 0.30 |
| Allocations (eq %) | AGG 90 · NEUT 55 · CONF 15 · DEF 85 |
| Challenge window assigned | 2021-01-01 → 2024-12-31 |


## 1. Imports and configuration

In [ ]:
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

warnings.filterwarnings("ignore")

# Plot styling
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "font.size": 10,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# Paths — adjust DATA_DIR to wherever your CSVs live
DATA_DIR = Path("./data")            # bundled CSVs (VFINX_daily.csv etc.)
OUT_DIR  = Path("./karthik1_outputs") # exports land here
OUT_DIR.mkdir(exist_ok=True)

print(f"Data directory:   {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")


## 2. Strategy configuration

All hyperparameters live in this single dictionary so the rest of the notebook is parameter-free.


In [ ]:
CONFIG = {
    # Asset pair
    "equity":       "VFINX",
    "fixed_income": "VBMFX",

    # Fit window (in-sample training period)
    "fit_start": "1995-01-01",
    "fit_end":   "1998-12-31",

    # Forward window over which we measure DD and UP events
    "forward_window_days": 63,

    # Drawdown model
    "dd_threshold_pct": 15.0,    # event = forward 63d max drawdown >= 15%
    "dd_prob_cutoff":   0.40,    # signal = P(DD) >= 0.40

    # Upside model
    "up_threshold_pct": 8.0,     # event = forward 63d max cumulative return >= 8%
    "up_prob_cutoff":   0.30,    # signal = P(UP) >= 0.30 (deliberately low)

    # Four-state equity allocations (rest goes to fixed income)
    "alloc_aggressive": 0.90,    # UP=1, DD=0
    "alloc_neutral":    0.55,    # UP=0, DD=0
    "alloc_conflicted": 0.15,    # UP=1, DD=1 (defense when models disagree)
    "alloc_defensive":  0.85,    # UP=0, DD=1 (CONTRARIAN — DD-only fires near bottoms)

    # Logit hyperparameters
    "logit_C":        1.0,
    "logit_penalty":  "l1",
    "logit_solver":   "liblinear",
    "logit_class_weight": "balanced",
    "logit_max_iter": 2000,
    "logit_random_state": 42,

    # Strategy name (used for export filenames)
    "strategy_name": "Karthik-1",
}

# Pretty-print config
print(json.dumps(CONFIG, indent=2))


## 3. Load price data

Six Vanguard mutual funds with continuous daily history from 1990-01-02 onwards. Using mutual funds (not ETFs) avoids survivorship bias — none were added or removed mid-sample.


In [ ]:
TICKERS = ["VFINX", "VEXMX", "VWIGX", "VBMFX", "VUSTX", "VWEHX"]

prices = {}
for t in TICKERS:
    df = pd.read_csv(DATA_DIR / f"{t}_daily.csv", parse_dates=["Date"])
    prices[t] = df.set_index("Date")["Close"].astype(float)

px = pd.DataFrame(prices).sort_index().dropna()
log_rets = np.log(px / px.shift(1)).dropna()
simple_rets = (np.exp(log_rets) - 1)

print(f"Price panel: {px.index[0].date()} → {px.index[-1].date()}  ({len(px):,} rows)")
print(f"Log returns: {len(log_rets):,} obs (after dropping first day)")
print()
print("First 3 rows of log returns:")
log_rets.head(3).round(4)


## 4. Feature engineering (FF4 + VIX proxies)

We construct seven raw daily features inspired by Fama-French 4-factor + VIX:

| Feature | Formula | Interpretation |
|---|---|---|
| Mkt | VFINX log return | Market |
| SMB_proxy | VEXMX − VFINX | Size premium proxy (extended − S&P 500) |
| INTL_US | VWIGX − VFINX | International vs US |
| Mom | 252-day cum return − 21-day cum return | Skip-month momentum |
| VIX_proxy | 21-day rolling vol × √252 | Realized volatility |
| Term | VUSTX − VBMFX | Term premium (long Treasury minus aggregate) |
| Credit | VWEHX − VUSTX | Credit premium (high yield minus Treasury) |

Each feature is then expanded into three forms — raw, 5-day rolling mean, 21-day rolling mean — giving **21 features total**.

**Critical: every feature is lagged by 1 day before joining with the target.** This eliminates lookahead bias.


In [ ]:
def build_features(log_rets):
    """Construct the 21-feature panel from daily log returns."""
    feat = pd.DataFrame(index=log_rets.index)
    feat["Mkt"]       = log_rets["VFINX"]
    feat["SMB_proxy"] = log_rets["VEXMX"] - log_rets["VFINX"]
    feat["INTL_US"]   = log_rets["VWIGX"] - log_rets["VFINX"]
    feat["Mom"]       = log_rets["VFINX"].rolling(252).sum() - log_rets["VFINX"].rolling(21).sum()
    feat["VIX_proxy"] = log_rets["VFINX"].rolling(21).std() * np.sqrt(252)
    feat["Term"]      = log_rets["VUSTX"] - log_rets["VBMFX"]
    feat["Credit"]    = log_rets["VWEHX"] - log_rets["VUSTX"]

    # Add 5-day and 21-day rolling means of each raw feature
    raw_cols = list(feat.columns)
    for c in raw_cols:
        feat[f"{c}_5d"]  = feat[c].rolling(5).mean()
        feat[f"{c}_21d"] = feat[c].rolling(21).mean()

    # CRITICAL: lag by 1 day to prevent lookahead
    feat = feat.shift(1).dropna()
    return feat

features = build_features(log_rets)
FEAT_COLS = list(features.columns)
print(f"Features panel: {features.index[0].date()} → {features.index[-1].date()}  ({len(features):,} rows)")
print(f"Feature columns ({len(FEAT_COLS)}):")
for i, c in enumerate(FEAT_COLS):
    print(f"  {i+1:2d}. {c}")


## 5. Target construction

Both targets use a **63-day forward window** (≈ one calendar quarter).

- **DD target** = 1 if max peak-to-trough drawdown of VFINX cumulative price over days [t, t+63] ≤ −15%
- **UP target** = 1 if max cumulative return of VFINX over days [t+1, t+63] ≥ +8%

The last 63 rows of the panel cannot be labeled and are dropped.


In [ ]:
def build_dd_target(price_index, threshold_pct, window=63):
    """For each t, label = 1 if any peak-to-trough drawdown over [t, t+window] <= -threshold."""
    arr = price_index.values
    out = np.full(len(arr), np.nan)
    thr = -(threshold_pct / 100.0)
    for i in range(len(arr) - window):
        seg = arr[i:i + window]
        rolling_max = np.maximum.accumulate(seg)
        worst_dd = ((seg / rolling_max) - 1).min()
        out[i] = 1.0 if worst_dd <= thr else 0.0
    return pd.Series(out, index=price_index.index, name="DD")


def build_up_target(simple_returns, threshold_pct, window=63):
    """For each t, label = 1 if max cumulative return over [t+1, t+window] >= threshold."""
    arr = simple_returns.values
    out = np.full(len(arr), np.nan)
    thr = threshold_pct / 100.0
    for i in range(len(arr) - window):
        cum = np.cumsum(arr[i + 1:i + 1 + window])
        out[i] = 1.0 if cum.max() >= thr else 0.0
    return pd.Series(out, index=simple_returns.index, name="UP")


# Cumulative VFINX index for DD target
mkt_idx = (1 + log_rets["VFINX"]).cumprod()

y_dd = build_dd_target(mkt_idx, CONFIG["dd_threshold_pct"], CONFIG["forward_window_days"])
y_up = build_up_target(log_rets["VFINX"], CONFIG["up_threshold_pct"], CONFIG["forward_window_days"])

print(f"DD target ({CONFIG['dd_threshold_pct']}%): base rate over full panel = {y_dd.dropna().mean():.1%}")
print(f"UP target ({CONFIG['up_threshold_pct']}%): base rate over full panel = {y_up.dropna().mean():.1%}")


## 6. Fit the two logit models on the in-sample window

Both models share the same feature set and hyperparameters. The L1 penalty self-prunes uninformative features.


In [ ]:
def fit_logit(features, target, fit_start, fit_end, config):
    """Fit an L1-regularized logit on the in-sample window. Returns (model, panel, in_sample_mask)."""
    panel = features.join(target.rename("y"), how="inner").dropna()
    mask_is = (panel.index >= fit_start) & (panel.index <= fit_end)
    train = panel.loc[mask_is]

    if train["y"].mean() < 0.005 or train["y"].mean() > 0.995:
        raise ValueError(f"In-sample target is degenerate (base rate {train['y'].mean():.1%})")

    clf = LogisticRegression(
        penalty=config["logit_penalty"],
        solver=config["logit_solver"],
        C=config["logit_C"],
        class_weight=config["logit_class_weight"],
        max_iter=config["logit_max_iter"],
        random_state=config["logit_random_state"],
    )
    clf.fit(train[FEAT_COLS], train["y"].astype(int))
    return clf, panel, mask_is


# Fit DD model
clf_dd, panel_dd, mask_is_dd = fit_logit(features, y_dd, CONFIG["fit_start"], CONFIG["fit_end"], CONFIG)
print(f"DD logit trained on {mask_is_dd.sum():,} in-sample obs (base rate {panel_dd.loc[mask_is_dd, 'y'].mean():.1%})")

# Fit UP model
clf_up, panel_up, mask_is_up = fit_logit(features, y_up, CONFIG["fit_start"], CONFIG["fit_end"], CONFIG)
print(f"UP logit trained on {mask_is_up.sum():,} in-sample obs (base rate {panel_up.loc[mask_is_up, 'y'].mean():.1%})")

# Predict probabilities on the full panel
prob_dd_full = pd.Series(clf_dd.predict_proba(panel_dd[FEAT_COLS])[:, 1], index=panel_dd.index, name="P_DD")
prob_up_full = pd.Series(clf_up.predict_proba(panel_up[FEAT_COLS])[:, 1], index=panel_up.index, name="P_UP")

print()
print("Probability summary statistics (in-sample):")
print(pd.DataFrame({
    "P_DD": prob_dd_full.loc[mask_is_dd].describe(),
    "P_UP": prob_up_full.loc[mask_is_up].describe(),
}).round(3))


## 7. Model diagnostics — coefficients

The L1 penalty drives many coefficients to exactly zero. The non-zero coefficients tell you which features the model actually uses.


In [ ]:
def show_coefficients(model, feat_cols, title):
    coefs = pd.Series(model.coef_[0], index=feat_cols).sort_values()
    nonzero = coefs[coefs != 0]

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#B85042" if c < 0 else "#5A8C5C" for c in nonzero]
    ax.barh(range(len(nonzero)), nonzero.values, color=colors, edgecolor="white")
    ax.set_yticks(range(len(nonzero)))
    ax.set_yticklabels(nonzero.index, fontsize=9)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_xlabel("Log-odds coefficient")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    print(f"  Non-zero features: {(coefs != 0).sum()} of {len(coefs)}")
    print(f"  Intercept: {model.intercept_[0]:+.3f}")

show_coefficients(clf_dd, FEAT_COLS, "DD logit coefficients (in-sample fit)")
show_coefficients(clf_up, FEAT_COLS, "UP logit coefficients (in-sample fit)")


## 8. Confusion matrices — in-sample vs. out-of-sample

We deliberately split the OOS into **Pre-IS** (1991 → fit_start) and **Post-IS** (fit_end → end of data) to see if the model behaves differently before and after the training era.

This is where the **most important finding** of the project shows up: in-sample DD recall is 100%, OOS recall is 47.9%.


In [ ]:
def compute_confusion(y_true, prob, cutoff, label):
    """Compute confusion matrix and headline metrics at a given probability cutoff."""
    y_pred = (prob >= cutoff).astype(int)
    y_true = y_true.astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    n = len(y_true)
    return {
        "label": label,
        "n": n,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "base_rate": y_true.mean(),
    }


def split_panel(panel, prob, fit_start, fit_end):
    """Return three DataFrames: pre-IS, in-sample, post-IS."""
    full = panel.join(prob, how="inner").dropna()
    is_mask  = (full.index >= fit_start) & (full.index <= fit_end)
    pre_mask = full.index < fit_start
    post_mask = full.index > fit_end
    return full.loc[pre_mask], full.loc[is_mask], full.loc[post_mask]


# Drawdown model
pre_dd, is_dd, post_dd = split_panel(panel_dd, prob_dd_full, CONFIG["fit_start"], CONFIG["fit_end"])
dd_results = [
    compute_confusion(pre_dd["y"],  pre_dd["P_DD"],  CONFIG["dd_prob_cutoff"], f"Pre-IS  {pre_dd.index[0].date()} → {pre_dd.index[-1].date()}"),
    compute_confusion(is_dd["y"],   is_dd["P_DD"],   CONFIG["dd_prob_cutoff"], f"In-Sample  {is_dd.index[0].date()} → {is_dd.index[-1].date()}"),
    compute_confusion(post_dd["y"], post_dd["P_DD"], CONFIG["dd_prob_cutoff"], f"Post-IS  {post_dd.index[0].date()} → {post_dd.index[-1].date()}"),
]

# Upside model
pre_up, is_up, post_up = split_panel(panel_up, prob_up_full, CONFIG["fit_start"], CONFIG["fit_end"])
up_results = [
    compute_confusion(pre_up["y"],  pre_up["P_UP"],  CONFIG["up_prob_cutoff"], f"Pre-IS  {pre_up.index[0].date()} → {pre_up.index[-1].date()}"),
    compute_confusion(is_up["y"],   is_up["P_UP"],   CONFIG["up_prob_cutoff"], f"In-Sample  {is_up.index[0].date()} → {is_up.index[-1].date()}"),
    compute_confusion(post_up["y"], post_up["P_UP"], CONFIG["up_prob_cutoff"], f"Post-IS  {post_up.index[0].date()} → {post_up.index[-1].date()}"),
]

def print_metrics_table(results, model_name):
    print(f"\n=== {model_name} confusion-matrix metrics ===")
    df = pd.DataFrame(results)[["label", "n", "TN", "FP", "FN", "TP", "base_rate", "precision", "recall", "f1", "accuracy"]]
    df = df.set_index("label")
    print(df.round(3).to_string())

print_metrics_table(dd_results, "Drawdown model")
print_metrics_table(up_results, "Upside model")


In [ ]:
def plot_confusion_grid(results, model_name, suptitle):
    """Plot 3 confusion matrices side by side (Pre-IS, IS, Post-IS)."""
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, r in zip(axes, results):
        cm = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]])
        im = ax.imshow(cm, cmap="Greens", aspect="auto")
        for (i, j), v in np.ndenumerate(cm):
            color = "white" if cm[i, j] > cm.max() / 2 else "#1A1F2E"
            label = ["TN", "FP", "FN", "TP"][i * 2 + j]
            ax.text(j, i - 0.15, label, ha="center", va="center", color=color, fontsize=9, fontweight="bold")
            ax.text(j, i + 0.15, str(int(v)), ha="center", va="center", color=color, fontsize=14, fontweight="bold")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred 0", "Pred 1"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Act 0", "Act 1"])
        ax.set_title(r["label"], fontsize=10)
        ax.set_xlabel(f"Prec {r['precision']:.2f} · Rec {r['recall']:.2f} · F1 {r['f1']:.2f}", fontsize=9)
    fig.suptitle(suptitle, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_confusion_grid(dd_results, "DD", f"Drawdown model · DD ≥ {CONFIG['dd_threshold_pct']:.0f}% · cutoff {CONFIG['dd_prob_cutoff']:.2f}")
plot_confusion_grid(up_results, "UP", f"Upside model · UP ≥ {CONFIG['up_threshold_pct']:.0f}% · cutoff {CONFIG['up_prob_cutoff']:.2f}")


## 9. AUC across splits

ROC AUC is a cleaner measure of discriminative power than accuracy because it doesn't depend on the cutoff.

**Watch for: DD AUC dropping below 0.50 on Post-IS data — that means the DD model is anti-predictive on unseen data.** This is the project's most important finding.


In [ ]:
def safe_auc(y_true, prob):
    if len(set(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, prob)


auc_table = pd.DataFrame({
    "DD AUC": [
        safe_auc(pre_dd["y"],  pre_dd["P_DD"]),
        safe_auc(is_dd["y"],   is_dd["P_DD"]),
        safe_auc(post_dd["y"], post_dd["P_DD"]),
    ],
    "UP AUC": [
        safe_auc(pre_up["y"],  pre_up["P_UP"]),
        safe_auc(is_up["y"],   is_up["P_UP"]),
        safe_auc(post_up["y"], post_up["P_UP"]),
    ],
}, index=["Pre-IS", "In-Sample", "Post-IS"]).round(3)
print(auc_table.to_string())
print()
print("Interpretation:")
print("  - In-sample AUCs are biased upward (model fitted on this data)")
print("  - Pre-IS and Post-IS AUCs are honest OOS measures")
print("  - AUC < 0.50 means the model is anti-predictive on that period")


## 10. Strategy simulation engine

This implements the same logic as the FE747 sim engine:

1. Compute daily DD signal and UP signal from probability cutoffs.
2. Map (UP, DD) to one of four states.
3. Pick the equity allocation for that state.
4. **Apply monthly rebalancing**: signal at month-end is held flat through the next month.
5. Compute daily strategy return = `w × eq_return + (1-w) × fi_return`.
6. Compute benchmark = static 50/50 of the chosen pair.

Run on any date range to test on different windows.


In [ ]:
def simulate_strategy(prob_dd, prob_up, simple_returns, eq_ticker, fi_ticker, config):
    """
    Run the four-state allocation simulation.

    Returns:
        DataFrame with columns:
            date, p_dd, p_up, signal_dd, signal_up, state, alloc_eq,
            eq_ret, fi_ret, strat_ret, bmk_ret, cum_strat, cum_bmk, rel_wealth
    """
    # Align everything
    common = prob_dd.index.intersection(prob_up.index).intersection(simple_returns.index).sort_values()
    p_dd = prob_dd.loc[common].values
    p_up = prob_up.loc[common].values
    eq_r = simple_returns[eq_ticker].loc[common].values
    fi_r = simple_returns[fi_ticker].loc[common].values

    # Signals
    s_dd = (p_dd >= config["dd_prob_cutoff"]).astype(int)
    s_up = (p_up >= config["up_prob_cutoff"]).astype(int)

    # State and alloc
    states = np.where((s_up == 1) & (s_dd == 0), "AGG",
             np.where((s_up == 0) & (s_dd == 0), "NEUT",
             np.where((s_up == 1) & (s_dd == 1), "CONF", "DEF")))
    alloc_map = {"AGG": config["alloc_aggressive"], "NEUT": config["alloc_neutral"],
                 "CONF": config["alloc_conflicted"], "DEF": config["alloc_defensive"]}
    daily_alloc = np.array([alloc_map[s] for s in states])

    # Monthly rebalance: state at month-end carries through to next month-end
    ym = pd.Series(common.year * 100 + common.month)
    is_month_end = (ym != ym.shift(-1)).fillna(True).values
    active_alloc = np.zeros(len(common))
    active_state = np.empty(len(common), dtype=object)
    cur_alloc, cur_state = daily_alloc[0], states[0]
    for i in range(len(common)):
        active_alloc[i] = cur_alloc
        active_state[i] = cur_state
        if is_month_end[i]:
            cur_alloc = daily_alloc[i]
            cur_state = states[i]

    strat_ret = active_alloc * eq_r + (1 - active_alloc) * fi_r
    bmk_ret   = 0.5 * eq_r + 0.5 * fi_r

    df = pd.DataFrame({
        "date":      common,
        "p_dd":      p_dd,
        "p_up":      p_up,
        "signal_dd": s_dd,
        "signal_up": s_up,
        "state":     active_state,
        "alloc_eq":  active_alloc,
        "eq_ret":    eq_r,
        "fi_ret":    fi_r,
        "strat_ret": strat_ret,
        "bmk_ret":   bmk_ret,
    }).set_index("date")
    df["cum_strat"]  = (1 + df["strat_ret"]).cumprod()
    df["cum_bmk"]    = (1 + df["bmk_ret"]).cumprod()
    df["rel_wealth"] = df["cum_strat"] - df["cum_bmk"]
    return df


def perf_summary(returns):
    """Compute CAGR, Sharpe, MaxDD, total return for a daily return series."""
    if len(returns) < 5:
        return {"CAGR": np.nan, "Sharpe": np.nan, "MaxDD": np.nan, "Total": np.nan}
    eq = (1 + returns).cumprod()
    yrs = len(returns) / 252
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    vol = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() * 252) / vol if vol > 0 else 0
    dd = (eq / eq.cummax() - 1).min()
    return {"CAGR": cagr, "Sharpe": sharpe, "MaxDD": dd, "Total": eq.iloc[-1] - 1}


# Run the full strategy across all available data
sim = simulate_strategy(prob_dd_full, prob_up_full, simple_rets,
                         CONFIG["equity"], CONFIG["fixed_income"], CONFIG)
print(f"Simulation panel: {sim.index[0].date()} → {sim.index[-1].date()}  ({len(sim):,} days)")
print()
print("State distribution across full panel:")
print(sim["state"].value_counts(normalize=True).map(lambda x: f"{x:.1%}").to_string())
print()
print(f"Average daily equity weight: {sim['alloc_eq'].mean():.1%}")


## 11. Performance — showcase window (1995-1998)

This is the in-sample window. Performance here is biased upward because the model was fitted on this data, but it shows the strategy under ideal conditions.


In [ ]:
def windowed_perf(sim, start, end, label):
    mask = (sim.index >= start) & (sim.index <= end)
    if mask.sum() < 5:
        return None
    ps = perf_summary(sim.loc[mask, "strat_ret"])
    pb = perf_summary(sim.loc[mask, "bmk_ret"])
    return {
        "Window": label,
        "Strat CAGR":  f"{ps['CAGR']:+.2%}",
        "Strat Sharpe": f"{ps['Sharpe']:.2f}",
        "Strat MaxDD":  f"{ps['MaxDD']:+.2%}",
        "Strat Total":  f"{ps['Total']:+.1%}",
        "Bmk CAGR":     f"{pb['CAGR']:+.2%}",
        "Bmk MaxDD":    f"{pb['MaxDD']:+.2%}",
        "Alpha (CAGR)": f"{ps['CAGR'] - pb['CAGR']:+.2%}",
    }


showcase = windowed_perf(sim, "1995-01-01", "1998-12-31", "Showcase 1995-1998 (in-sample)")
print(pd.Series(showcase).to_string())


In [ ]:
def plot_equity_curves(sim, start, end, title):
    df = sim.loc[(sim.index >= start) & (sim.index <= end)].copy()
    df["cum_strat_norm"] = df["cum_strat"] / df["cum_strat"].iloc[0]
    df["cum_bmk_norm"]   = df["cum_bmk"]   / df["cum_bmk"].iloc[0]

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                              gridspec_kw={"height_ratios": [2.2, 1]})
    ax = axes[0]
    ax.plot(df.index, df["cum_strat_norm"], lw=2.5, color="#2D5F5D", label="Strategy")
    ax.plot(df.index, df["cum_bmk_norm"],   lw=1.5, color="#888780", ls="--", label="Benchmark (50/50)")
    ax.set_ylabel("Growth of $1")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=10)

    ax = axes[1]
    rel_pct = (df["cum_strat_norm"] - df["cum_bmk_norm"]) * 100
    ax.fill_between(df.index, rel_pct, 0, where=(rel_pct >= 0), color="#5A8C5C", alpha=0.4)
    ax.fill_between(df.index, rel_pct, 0, where=(rel_pct < 0),  color="#B85042", alpha=0.4)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_ylabel("Relative wealth (%)")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()


plot_equity_curves(sim, "1995-01-01", "1998-12-31",
                   "Showcase window 1995-1998 — Strategy vs 50/50 benchmark")


## 12. Performance — challenge window (2021-2024)

The 2021-2024 window is the one we assigned to opponents — bond bear + stock bear simultaneously, the only stress event in the dataset where stock-bond correlation flipped positive.

**Honest expectation**: our own strategy survives here (+1.3% CAGR alpha) but doesn't dominate. Opponents using VUSTX-based strategies get destroyed.


In [ ]:
# Year-by-year and full-window performance
periods = [
    ("Showcase 1995-1998 (in-sample)", "1995-01-01", "1998-12-31"),
    ("Pre-IS 1991-1994 (true OOS)",    "1991-01-01", "1994-12-31"),
    ("Adversarial 2021-2024",          "2021-01-01", "2024-12-31"),
    ("    2021 only",                  "2021-01-01", "2021-12-31"),
    ("    2022 only",                  "2022-01-01", "2022-12-31"),
    ("    2023 only",                  "2023-01-01", "2023-12-31"),
    ("    2024 only",                  "2024-01-01", "2024-12-31"),
    ("Full panel",                     str(sim.index[0].date()), str(sim.index[-1].date())),
]
rows = [windowed_perf(sim, s, e, lbl) for lbl, s, e in periods]
rows = [r for r in rows if r is not None]
perf_df = pd.DataFrame(rows).set_index("Window")
print(perf_df.to_string())


In [ ]:
plot_equity_curves(sim, "2021-01-01", "2024-12-31",
                   "Adversarial window 2021-2024 — Strategy vs 50/50 benchmark")


## 13. State attribution by quarter

Where did each state actually fire? This is what reveals the **true** mechanism of the strategy:

- **Defensive (DEF=85% equity)** only fires when the DD signal is on AND the UP signal is off
- In strongly trending markets (2022 bear, 2023 recovery), the UP signal stays on through volatility, so the strategy hits Conflicted (CONF=15%) instead
- DEF protects against false-alarm wobbles in bull markets (2021); AGG protects through V-shaped recoveries (2023 Q1)


In [ ]:
def plot_state_attribution(sim, start, end, title):
    df = sim.loc[(sim.index >= start) & (sim.index <= end)].copy()
    df["q"] = df.index.to_period("Q").astype(str)
    qstates = df.groupby("q")["state"].value_counts(normalize=True).unstack(fill_value=0)
    for s in ["AGG", "NEUT", "CONF", "DEF"]:
        if s not in qstates.columns: qstates[s] = 0
    qstates = qstates[["AGG", "NEUT", "CONF", "DEF"]] * 100

    fig, ax = plt.subplots(figsize=(13, 5))
    colors = {"AGG": "#5A8C5C", "NEUT": "#888780", "CONF": "#BA7517", "DEF": "#2D5F5D"}
    labels = {"AGG": "Aggressive (90% eq)", "NEUT": "Neutral (55% eq)",
              "CONF": "Conflicted (15% eq)", "DEF": "Defensive (85% eq)"}
    bottom = np.zeros(len(qstates))
    for s in ["AGG", "NEUT", "CONF", "DEF"]:
        ax.bar(qstates.index, qstates[s], bottom=bottom, color=colors[s],
                label=labels[s], width=0.7, edgecolor="white", linewidth=0.5)
        bottom += qstates[s].values
    ax.set_ylabel("% of trading days")
    ax.set_ylim(0, 100)
    ax.set_title(title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=9, frameon=False)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.tight_layout()
    plt.show()


plot_state_attribution(sim, "2021-01-01", "2024-12-31",
                       "Quarterly state attribution — 2021-2024 adversarial window")


## 14. Export model JSONs and probability CSVs

These are the files the FE747 sim engine consumes. Drop them into the engine's models folder to reproduce the strategy in the official environment.


In [ ]:
def export_model_json(model, target_label, threshold_pct, prob_cutoff, fit_start, fit_end, feat_cols, out_path):
    payload = {
        "strategy": CONFIG["strategy_name"],
        "target": target_label,
        "fit_window": {
            "start": fit_start,
            "end":   fit_end,
        },
        "thresholds": {
            "threshold_pct":      float(threshold_pct),
            "probability_cutoff": float(prob_cutoff),
        },
        "model": {
            "type": "logistic_regression",
            "penalty": CONFIG["logit_penalty"],
            "C":       CONFIG["logit_C"],
            "intercept": float(model.intercept_[0]),
            "coefficients": {f: float(c) for f, c in zip(feat_cols, model.coef_[0])},
        },
    }
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"  Wrote {out_path}")


export_model_json(
    clf_dd, target_label="drawdown",
    threshold_pct=CONFIG["dd_threshold_pct"],
    prob_cutoff=CONFIG["dd_prob_cutoff"],
    fit_start=CONFIG["fit_start"], fit_end=CONFIG["fit_end"],
    feat_cols=FEAT_COLS,
    out_path=OUT_DIR / f"{CONFIG['strategy_name']}_drawdown_model.json",
)
export_model_json(
    clf_up, target_label="upside",
    threshold_pct=CONFIG["up_threshold_pct"],
    prob_cutoff=CONFIG["up_prob_cutoff"],
    fit_start=CONFIG["fit_start"], fit_end=CONFIG["fit_end"],
    feat_cols=FEAT_COLS,
    out_path=OUT_DIR / f"{CONFIG['strategy_name']}_upside_model.json",
)

# Probability CSVs (full panel)
prob_dd_full.to_frame().to_csv(OUT_DIR / f"{CONFIG['strategy_name']}_drawdown_probs.csv")
prob_up_full.to_frame().to_csv(OUT_DIR / f"{CONFIG['strategy_name']}_upside_probs.csv")
print(f"  Wrote {OUT_DIR / (CONFIG['strategy_name'] + '_drawdown_probs.csv')}")
print(f"  Wrote {OUT_DIR / (CONFIG['strategy_name'] + '_upside_probs.csv')}")

# Strategy config JSON
strat_config = {
    "strategy_name":  CONFIG["strategy_name"],
    "equity":         CONFIG["equity"],
    "fixed_income":   CONFIG["fixed_income"],
    "allocations":    {
        "AGGRESSIVE": CONFIG["alloc_aggressive"],
        "NEUTRAL":    CONFIG["alloc_neutral"],
        "CONFLICTED": CONFIG["alloc_conflicted"],
        "DEFENSIVE":  CONFIG["alloc_defensive"],
    },
    "rebalance_frequency": "monthly",
    "challenge_window_assigned": {"start": "2021-01-01", "end": "2024-12-31"},
}
with open(OUT_DIR / f"{CONFIG['strategy_name']}_strategy_config.json", "w") as f:
    json.dump(strat_config, f, indent=2)
print(f"  Wrote {OUT_DIR / (CONFIG['strategy_name'] + '_strategy_config.json')}")

# Daily simulation CSV (full panel)
sim.to_csv(OUT_DIR / f"{CONFIG['strategy_name']}_daily_simulation.csv")
print(f"  Wrote {OUT_DIR / (CONFIG['strategy_name'] + '_daily_simulation.csv')}")

print()
print("All exports complete. Files in:", OUT_DIR.resolve())


## 15. Honest caveats

This notebook reproduces the strategy as submitted, but the team should be aware of the following before defending it:

1. **In-sample DD recall (100%) is an overfit artifact.** Out-of-sample recall is 47.9% on 6,767 unseen days. The strategy's drawdown protection comes from allocation design (DEF=85% equity), not from signal accuracy.

2. **The Mom_21d coefficient of ~+1500 is unusually large.** This means the L1 penalty did not fully constrain the model in this configuration. Lowering `logit_C` from 1.0 to 0.1 would produce more interpretable coefficients but might reduce in-sample fit.

3. **The Upside model is barely better than random** at the 8% threshold (in-sample accuracy 57.6% vs base rate 56.5%). It functions as a "default on" filter rather than a genuine prediction.

4. **No transaction costs modeled.** Real-world deployment with monthly rebalancing would face approximately 1–2% annual drag from frictions.

5. **The 1995-1998 fit window over-represents fast V-shaped crashes** (Asian, LTCM). The model has never seen a slow grinding bear market like 2000-02 or 2008-09, which is why DD recall degrades severely on those periods.

For a v2 strategy, the most impactful change would be **retraining on a longer window (e.g., 2002-2018)** that contains five distinct crash regimes instead of two. Expected outcome: smaller showcase numbers, much better OOS generalization.

---

*End of notebook.*
